IMPORTS & SETUP

In [1]:
import pandas as pd
import sqlite3
import json
import os

# File paths
patents_path = 'g_patent\\g_patent.tsv'
abstracts_path = 'g_patent_abstract\\g_patent_abstract.tsv'
inventors_path = 'g_inventor_disambiguated\\g_inventor_disambiguated.tsv'
patent_inventor_path = 'g_persistent_inventor\\g_persistent_inventor.tsv'
companies_path = 'g_assignee_disambiguated\\g_assignee_disambiguated.tsv'
patent_company_path = 'g_persistent_assignee\\g_persistent_assignee.tsv'
locations_path = 'g_location_disambiguated\\g_location_disambiguated.tsv'

os.makedirs('working', exist_ok=True)

print("Setup complete")

Setup complete


CHUNK LOADER

In [2]:
def load_large_csv(path, cols, nrows=50000):
    return pd.concat(
        pd.read_csv(
            path,
            sep='\t',
            usecols=cols,
            chunksize=50000,
            nrows=nrows
        ),
        ignore_index=True
    )

print("Loader ready")

Loader ready


LOAD DATA

In [3]:
patents = load_large_csv(
    patents_path,
    ['patent_id','patent_title','patent_date']
)

abstracts = load_large_csv(
    abstracts_path,
    ['patent_id','patent_abstract']
)

inventors = load_large_csv(
    inventors_path,
    [
        'patent_id',
        'inventor_id',
        'disambig_inventor_name_first',
        'disambig_inventor_name_last',
        'location_id'
    ]
)

companies = load_large_csv(
    companies_path,
    [
        'patent_id',
        'assignee_id',
        'disambig_assignee_organization'
    ]
)

locations = load_large_csv(
    locations_path,
    ['location_id','disambig_country']
)

print("All data loaded correctly")

All data loaded correctly


CLEAN DATA

In [4]:
# Rename
locations.rename(columns={'disambig_country':'country'}, inplace=True)

companies.rename(columns={
    'assignee_id':'company_id',
    'disambig_assignee_organization':'name'
}, inplace=True)

# -------------------------
# PATENTS
patents_clean = patents.merge(abstracts, on='patent_id', how='left')

patents_clean['year'] = pd.to_datetime(
    patents_clean['patent_date'],
    errors='coerce'
).dt.year

patents_clean['year'] = patents_clean['year'].fillna(0).astype(int)

# -------------------------
# INVENTORS
inventors['name'] = (
    inventors['disambig_inventor_name_first'].fillna('') + ' ' +
    inventors['disambig_inventor_name_last'].fillna('')
)

inventors_clean = inventors.merge(
    locations,
    on='location_id',
    how='left'
)[['inventor_id','name','country']]

inventors_clean = inventors_clean.drop_duplicates(subset=['inventor_id'])

# -------------------------
# COMPANIES
companies_clean = companies[['company_id','name']].dropna()
companies_clean = companies_clean.drop_duplicates(subset=['company_id'])

print("Cleaning complete")

Cleaning complete


RELATIONSHIPS

In [5]:
# patent → inventor
rel_inventor = inventors[['patent_id','inventor_id']]

# patent → company
rel_company = companies[['patent_id','company_id']]

# combine
relationships = rel_inventor.merge(
    rel_company,
    on='patent_id',
    how='inner'
)

relationships = relationships.dropna()

print("Relationships created")

Relationships created


SAVE CLEAN FILES

In [6]:
patents_clean.to_csv('working\\clean_patents.csv', index=False)
inventors_clean.to_csv('working\\clean_inventors.csv', index=False)
companies_clean.to_csv('working\\clean_companies.csv', index=False)
relationships.to_csv('working\\relationships.csv', index=False)

print('Saved cleaned files')

Saved cleaned files


LOAD INTO DATABASE

In [7]:
conn = sqlite3.connect('working/patents.db')

patents_clean.to_sql('patents', conn, if_exists='replace', index=False)
inventors_clean.to_sql('inventors', conn, if_exists='replace', index=False)
companies_clean.to_sql('companies', conn, if_exists='replace', index=False)
relationships.to_sql('relationships', conn, if_exists='replace', index=False)

print("Database ready")

Database ready


SQL QUERIES

Top Inventors

In [8]:
top_inventors = pd.read_sql_query("""
SELECT i.name, COUNT(*) AS total_patents
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
GROUP BY i.name
ORDER BY total_patents DESC
LIMIT 10
""", conn)

top_inventors

,name,total_patents
0,Dong Min KANG,2
1,Zuzana Kossaczka,1
2,Zhouyue Pi,1
3,Zhibiao Zhao,1
4,Zhenyu Li,1
5,Yutaka Mizuno,1
6,Yutaka Miki,1
7,Yurie Ota,1
8,Yunjung Yi,1
9,Yun-song Jeong,1


Top Companies

In [9]:
top_companies = pd.read_sql_query("""
SELECT c.name, COUNT(*) AS total_patents
FROM relationships r
JOIN companies c ON r.company_id = c.company_id
GROUP BY c.name
ORDER BY total_patents DESC
LIMIT 10
""", conn)

top_companies

,name,total_patents
0,"SAMSUNG DISPLAY CO., LTD.",9
1,International Business Machines Corporation,7
2,SEIKO EPSON CORPORATION,3
3,"Micron Technology, Inc.",3
4,LG ELECTRONICS INC.,3
5,Kabushiki Kaisha Toshiba,3
6,"HONDA MOTOR CO., LTD.",3
7,GOOGLE LLC,3
8,"Comcast Cable Communications, LLC",3
9,"Boston Scientific Scimed, Inc.",3


Top Countries

In [10]:
top_countries = pd.read_sql_query("""
SELECT country, COUNT(*) AS total
FROM inventors
GROUP BY country
ORDER BY total DESC
LIMIT 10
""", conn)

top_countries

,country,total
0,None,24089
1,US,9020
2,JP,4107
3,DE,2383
4,KR,1753
5,CN,1314
6,FR,878
7,GB,699
8,CA,560
9,IL,421


Trends

In [11]:
trends = pd.read_sql_query("""
SELECT year, COUNT(*) AS total_patents
FROM patents
GROUP BY year
ORDER BY year
""", conn)

trends

,year,total_patents
0,2018,50000


JOIN

In [12]:
joined_data = pd.read_sql_query("""
SELECT p.patent_title, i.name, c.name
FROM relationships r
JOIN patents p ON r.patent_id = p.patent_id
JOIN inventors i ON r.inventor_id = i.inventor_id
JOIN companies c ON r.company_id = c.company_id
LIMIT 20
""", conn)

joined_data

,patent_title,name,name
0,Semiconductor device and manufacturing method ...,Shinichi Watanuki,Renesas Electronics Corporation
1,Electronic device and method for disassembling...,Ji-Woo Lee,"SAMSUNG DISPLAY CO., LTD."
2,Apparatus and method for communication using w...,Young Tack Hong,"SAMSUNG DISPLAY CO., LTD."


CTE

In [13]:
cte_query = pd.read_sql_query("""
WITH inventor_counts AS (
    SELECT inventor_id, COUNT(*) AS total
    FROM relationships
    GROUP BY inventor_id
)
SELECT *
FROM inventor_counts
WHERE total > 5
ORDER BY total DESC
""", conn)

cte_query

,inventor_id,total


Ranking

In [14]:
ranking = pd.read_sql_query("""
SELECT i.name,
       COUNT(*) AS total,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rank
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
GROUP BY i.name
LIMIT 10
""", conn)

ranking

,name,total,rank
0,Dong Min KANG,2,1
1,Zuzana Kossaczka,1,2
2,Zhouyue Pi,1,2
3,Zhibiao Zhao,1,2
4,Zhenyu Li,1,2
5,Yutaka Mizuno,1,2
6,Yutaka Miki,1,2
7,Yurie Ota,1,2
8,Yunjung Yi,1,2
9,Yun-song Jeong,1,2


EXPORT REPORTS

CSV

In [15]:
top_inventors.to_csv('working\\top_inventors.csv', index=False)
top_companies.to_csv('working\\top_companies.csv', index=False)
top_countries.to_csv('working\\country_trends.csv', index=False)


JSON

In [16]:
report = {
    "total_patents": int(len(patents_clean)),
    "top_inventors": top_inventors.to_dict(orient='records'),
    "top_companies": top_companies.to_dict(orient='records'),
    "top_countries": top_countries.to_dict(orient='records')
}

with open('working/report.json', 'w') as f:
    json.dump(report, f, indent=4)

Console Output

In [17]:
print("\nPATENT REPORT")
print(f"Total Patents: {len(patents_clean)}\n")

print("Top Inventors:")
print(top_inventors.to_string(index=False), "\n")

print("Top Companies:")
print(top_companies.to_string(index=False), "\n")

print("Top Countries:")
print(top_countries.to_string(index=False))


PATENT REPORT
Total Patents: 50000

Top Inventors:
            name  total_patents
   Dong Min KANG              2
Zuzana Kossaczka              1
      Zhouyue Pi              1
    Zhibiao Zhao              1
       Zhenyu Li              1
   Yutaka Mizuno              1
     Yutaka Miki              1
       Yurie Ota              1
      Yunjung Yi              1
  Yun-song Jeong              1 

Top Companies:
                                       name  total_patents
                  SAMSUNG DISPLAY CO., LTD.              9
International Business Machines Corporation              7
                    SEIKO EPSON CORPORATION              3
                    Micron Technology, Inc.              3
                        LG ELECTRONICS INC.              3
                   Kabushiki Kaisha Toshiba              3
                      HONDA MOTOR CO., LTD.              3
                                 GOOGLE LLC              3
          Comcast Cable Communications, LLC     